In [ ]:
#python

import pandas as pd
import numpy as np

from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression

import pickle



In [ ]:


with open('preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
scaler = data['scaler']

# Re-create scaled versions using the already-fitted scaler
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data loaded and scaled!")

Data loaded and scaled!


In [7]:
print(data.keys())

dict_keys(['X_train', 'X_test', 'y_train', 'y_test', 'scaler'])


In [9]:

#  Train Logistic Regression 
# class_weight='balanced' compensates for the 75/25 class imbalance
log_reg = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)

# Train the model: it learns weights from X_train and y_train
log_reg.fit(X_train_scaled, y_train)

print("Model trained!")

Model trained!


In [10]:
# Let's see the results:

coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Weight': log_reg.coef_[0]
}).sort_values('Weight', key=abs, ascending=False)

print(coef_df.to_string(index=False))

             Feature    Weight
      Lactate_mmol_L  1.287833
      Heart_Rate_BPM  1.194623
Arterial_Base_Excess -0.517049
         Injury_Type  0.432475
    Systolic_BP_mmHg  0.138944
         Shock_Index -0.006368


In [12]:

# Predict on test set
y_pred = log_reg.predict(X_test_scaled)
y_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

# ROC-AUC
auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC: {auc:.3f}")
print()

# Classification report
print(classification_report(y_test, y_pred, target_names=['Non-MT', 'MT']))
print()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

ROC-AUC: 0.880

              precision    recall  f1-score   support

      Non-MT       0.92      0.84      0.88       150
          MT       0.62      0.78      0.69        50

    accuracy                           0.82       200
   macro avg       0.77      0.81      0.78       200
weighted avg       0.84      0.82      0.83       200


Confusion Matrix:
[[126  24]
 [ 11  39]]


In [16]:
print("Predicted:", y_pred[:10])
print("Actual:   ", y_test.values[:10])

Predicted: [0 1 1 1 0 0 1 1 0 0]
Actual:    [0 1 1 1 0 0 1 1 0 0]


In [17]:
correct = (y_pred == y_test.values).sum()
total = len(y_test)
print(f"Correct predictions: {correct} out of {total}")
print(f"Accuracy: {correct/total:.3f}")

Correct predictions: 165 out of 200
Accuracy: 0.825
